In [ ]:
import pandas as pd
import numpy as np
from tabpfn import TabPFNClassifier
from sklearn.utils import shuffle

TARGET_COL = "Survived"
RANDOM_STATE = 42

df = pd.read_csv("titanic_dataset/train.csv")

df = shuffle(df, random_state=RANDOM_STATE).reset_index(drop=True)

drop_cols = [
    "PassengerId",
    "Name"
]

df = df.drop(columns=[f for f in drop_cols])

train_df_ds = df.iloc[:int(0.25 * len(df))].copy(deep=True)
test_with_gt_df = df.iloc[int(0.25 * len(df)):].copy(deep=True)

test_df = test_with_gt_df.drop(columns=[TARGET_COL])

train_df_ds.to_csv("train_split.csv", index=False)
test_with_gt_df.to_csv("test_with_gt_split.csv", index=False)

test_df.to_csv("test_split.csv", index=False)

print("All files generated.")


In [ ]:
print(train_df_ds.shape)
print(test_with_gt_df.shape)
print(test_df.shape)

In [ ]:
import os

# TabPFN checkpoints are downloaded from Hugging Face. Provide HF_TOKEN via the
# environment or a local .env file (see .env.example) - never hardcode it here.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is not set. Export it or add it to a .env file (see .env.example).")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tabpfn.finetuning import FinetunedTabPFNClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_auc_score, confusion_matrix, recall_score, ConfusionMatrixDisplay
import logging
logging.basicConfig(level=logging.INFO)

LORA_RANK = 8
LORA_ALPHA = 16

OUTPUT_DIR = Path("./tabpfn_checkpoints/run01")

TARGET_COL = "Survived"

TRAIN_FILES = ["train_split.csv"]

TEST_FILES = ["test_with_gt_split.csv"]

train_dfs = [pd.read_csv(f) for f in TRAIN_FILES]
train_df = pd.concat(train_dfs, axis=0, ignore_index=True)

X_train = train_df.drop(columns=[TARGET_COL]).values
y_train = train_df[TARGET_COL].values

clf = FinetunedTabPFNClassifier(
    device="cpu",
    epochs=30,
    learning_rate=1e-4,
    weight_decay=0.0,
    lora_r=8,
    lora_alpha=16.0,
    n_estimators_finetune=2,
    n_estimators_validation=2,
    n_estimators_final_inference=8,
)

clf.fit(X_train, y_train, output_dir=OUTPUT_DIR)


In [ ]:
sum(p.numel() for p in clf.finetuned_inference_classifier_.model_.parameters() if p.requires_grad)

In [ ]:
for n, p in clf.finetuned_inference_classifier_.model_.named_parameters():
    if p.requires_grad:
        print(n)